# Training Meditation vs Thinking Classifier

This notebook demonstrates the complete pipeline for training a neural network classifier to distinguish between meditation and thinking tasks based on HRV features extracted from ECG signals.

**Tasks:**
- med1breath (Label 1) vs think1 (Label 0)
- 98 total subjects, 4 tasks per subject

## 1. Setup and Imports

In [8]:
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load environment variables
env_path = Path('.').resolve() / '.env'
load_dotenv(env_path)

# Import EEG processing modules
from eeg_processor import Config, DataLoader, HRVFeatureExtractor, HRVClassifier

print(f"BIDS Root: {os.getenv('BIDS_ROOT')}")

BIDS Root: d:\Hackathons\EPFL_Life_Sciences_2026\ds003969


## 2. Configuration

In [9]:
config = Config(
    bids_root=os.getenv('BIDS_ROOT'),
    random_seed=42
)

MEDITATE_TASK = 'med1breath'  # Label 1 (CORRECTED task name)
THINK_TASK = 'think1'         # Label 0
MAX_SUBJECTS = 10

print(f"Tasks: {MEDITATE_TASK} (label 1) vs {THINK_TASK} (label 0)")

[OK] Configuration initialized with BIDS root: d:\Hackathons\EPFL_Life_Sciences_2026\ds003969
Tasks: med1breath (label 1) vs think1 (label 0)


## 3. Load Task-Specific Data

In [10]:
loader = DataLoader(config)
loader.explore_bids_structure()

loaded_data = {}
for subject_id in loader.subjects[:MAX_SUBJECTS]:
    print(f"Subject {subject_id}:")
    raw_med = loader.load_eeg_data(subject_id, session=None, task=MEDITATE_TASK)
    raw_think = loader.load_eeg_data(subject_id, session=None, task=THINK_TASK)
    
    if raw_med and raw_think:
        loaded_data[subject_id] = {MEDITATE_TASK: raw_med, THINK_TASK: raw_think}
        print(f"  ✓ Both tasks loaded")

print(f"\nLoaded {len(loaded_data)} subjects")

[OK] BIDS structure explored successfully
  - Subjects: 1 (['001'])
  - Sessions: 0 (None)
  - Tasks: 4 (['med1breath', 'med2', 'think1', 'think2'])
Subject 001:
Loading BIDS file: d:/Hackathons/EPFL_Life_Sciences_2026/ds003969/sub-001/eeg/sub-001_task-med1breath_eeg.bdf
[OK] Successfully loaded EEG data


d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: Did not find any events.tsv associated with sub-001_task-med1breath.

The search_str was "d:\Hackathons\EPFL_Life_Sciences_2026\ds003969\sub-001\**\eeg\sub-001*events.tsv"
  self.raw = read_raw_bids(bids_path=bids_path, verbose=verbose)
d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: The number of channels in the channels.tsv sidecar file (79) does not match the number of channels in the raw data file (80). Will not try to set channel names.
  self.raw = read_raw_bids(bids_path=bids_path, verbose=verbose)
d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: Unable to map the following column(s) to to MNE:
gender: m
group: htr
ethnicity: indian
first_session: meditation
sleep: 6
education: 0
years_of_practice: 3
notes: n/a
  self.raw = read_raw_bids(bids_path=bi

  - Shape: (80, 620544)
Loading BIDS file: d:/Hackathons/EPFL_Life_Sciences_2026/ds003969/sub-001/eeg/sub-001_task-think1_eeg.bdf
[OK] Successfully loaded EEG data


d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: Did not find any events.tsv associated with sub-001_task-think1.

The search_str was "d:\Hackathons\EPFL_Life_Sciences_2026\ds003969\sub-001\**\eeg\sub-001*events.tsv"
  self.raw = read_raw_bids(bids_path=bids_path, verbose=verbose)
d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: The number of channels in the channels.tsv sidecar file (79) does not match the number of channels in the raw data file (80). Will not try to set channel names.
  self.raw = read_raw_bids(bids_path=bids_path, verbose=verbose)
d:\Hackathons\EPFL_Life_Sciences_2026\EEG-Meditation-App\eeg\eeg_processor\data_loader.py:93: RuntimeWarning: Unable to map the following column(s) to to MNE:
gender: m
group: htr
ethnicity: indian
first_session: meditation
sleep: 6
education: 0
years_of_practice: 3
notes: n/a
  self.raw = read_raw_bids(bids_path=bids_p

  - Shape: (80, 620544)
  ✓ Both tasks loaded

Loaded 1 subjects


## 4. Extract HRV Features

In [ ]:
def extract_hrv(raw, ecg_channel='EXG7', sampling_rate=500):
    try:
        ecg_idx = raw.ch_names.index(ecg_channel)
        ecg_signal = raw.get_data(picks=ecg_idx)[0]
        extractor = HRVFeatureExtractor(sampling_rate=sampling_rate)
        features, _ = extractor.extract_with_interval(ecg_signal)
        return features
    except Exception as e:
        print(f"  Error: {e}")
        return None

all_features = []
all_labels = []
feature_names = ['rr_mean', 'sdnn', 'rmssd', 'pnn50', 'lf', 'hf', 'lf_hf', 'vlf']
channel_mapping = {'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog',
                   'EXG5': 'misc', 'EXG6': 'misc', 'EXG7': 'ecg'}

for subject_id, tasks_data in loaded_data.items():
    print(f"Subject {subject_id}:")
    
    for task_name, raw in tasks_data.items():
        loader.raw = raw
        loader.setup_montage(montage_name='biosemi64', on_missing='ignore')
        loader.set_channel_types(channel_mapping)
        
        features = extract_hrv(raw, sampling_rate=int(raw.info['sfreq']))
        if features:
            print(f"  ✓ {task_name}")
            all_features.append(list(features.values()))
            all_labels.append(1 if task_name == MEDITATE_TASK else 0)

X = np.array(all_features)
y = np.array(all_labels, dtype=int)

print(f"\nExtracted {len(X)} samples")
print(f"Classes: Think={np.sum(y==0)}, Meditate={np.sum(y==1)}")

## 5. Train/Val/Test Split

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

## 6. Train Model

In [ ]:
classifier = HRVClassifier(input_size=X_train.shape[1], hidden_sizes=[64, 32], learning_rate=0.001)

print(f"Training...\n")
history = classifier.train(X_train, y_train, X_val, y_val, epochs=100, batch_size=16, feature_names=feature_names)

## 7. Evaluate

In [ ]:
test_pred, test_probs = classifier.predict_with_proba(X_test)

accuracy = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred)
recall = recall_score(y_test, test_pred)
f1 = f1_score(y_test, test_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

## 8. Sample Predictions

In [ ]:
for i in range(min(5, len(X_test))):
    pred = "Think" if test_pred[i] == 0 else "Meditate"
    true = "Think" if y_test[i] == 0 else "Meditate"
    print(f"Sample {i+1}: True={true}, Pred={pred}, Think={test_probs[i][0]:.3f}, Med={test_probs[i][1]:.3f}")

## 9. Save Model

In [ ]:
model_dir = Path('.').resolve() / 'models' / 'meditation_classifier'
classifier.save(str(model_dir))
print(f"Model saved to {model_dir}")